# ComfyUI — klein-9B Skin Enhancer (clean rebuild)

> Add blockquote



**Before running:**
1. Runtime → Change runtime type → **GPU (A100/L4)**.
2. Add your **HF_TOKEN** in the 🔑 Secrets tab (and toggle notebook access ON).
3. Accept the klein-9B license once at https://huggingface.co/black-forest-labs/FLUX.2-klein-9B → 'Agree and access'.

Then run cells **1 → 4** top to bottom. Everything installs into `/content/ComfyUI` (local disk).
On a fresh runtime, just re-run all four — no manual file copying.

_Cell 4 stays running and embeds the ComfyUI UI inline — scroll down inside its output to use it._

In [1]:
# CELL 1 — Install ComfyUI + Manager + dependencies
import os
COMFY = '/content/ComfyUI'
if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
%cd {COMFY}
!pip install -q -r requirements.txt           # installs comfy_aimdo etc. (fixes the old ModuleNotFound)
if not os.path.isdir(f'{COMFY}/custom_nodes/comfyui-manager'):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager {COMFY}/custom_nodes/comfyui-manager
print('✅ ComfyUI + Manager ready')

Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 43284, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 43284 (delta 32), reused 17 (delta 17), pack-reused 43227 (from 3)
Receiving objects: 100% (43284/43284), 84.10 MiB | 18.45 MiB/s, done.
Resolving deltas: 100% (29357/29357), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 145.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 156

In [2]:
# CELL 2 — Custom nodes the workflow needs
import os, subprocess
CN = '/content/ComfyUI/custom_nodes'
repos = {
    # --- nodes REQUIRED by ai_skin_api.json ---
    'ComfyUI_Comfyroll_CustomNodes': 'https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes',  # CR Prompt Text, CR Text Concatenate (nodes 64,75)
    'ComfyUI-Easy-Use':              'https://github.com/yolain/ComfyUI-Easy-Use',                # easy imageColorMatch (node 83)
    'ComfyUI_EasySize':              'https://github.com/balu112121/ComfyUI_EasySize',            # EasySizeSimpleImage / 简单图像尺寸 (node 43)
    # --- kept from before (not used by this workflow, harmless) ---
    'ComfyUI_essentials':            'https://github.com/cubiq/ComfyUI_essentials',
    'comfyui_face_parsing':          'https://github.com/Ryuukeisyou/comfyui_face_parsing',
    'ComfyUI_LayerStyle':            'https://github.com/chflame163/ComfyUI_LayerStyle',
    'ComfyUI-Impact-Pack':           'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-KJNodes':               'https://github.com/kijai/ComfyUI-KJNodes',
    'ComfyUI-SeedVR2_VideoUpscaler': 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler',
}
for folder, url in repos.items():
    p = os.path.join(CN, folder)
    if os.path.isdir(p): print('⏭️', folder)
    else: print('⬇️', folder); subprocess.run(['git','clone','--depth','1',url,p], check=False)
for folder in repos:
    req = os.path.join(CN, folder, 'requirements.txt')
    if os.path.exists(req): subprocess.run(['pip','install','-q','-r',req], check=False)
print('✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)')


⬇️ ComfyUI_Comfyroll_CustomNodes
⬇️ ComfyUI-Easy-Use
⬇️ ComfyUI_EasySize
⬇️ ComfyUI_essentials
⬇️ comfyui_face_parsing
⬇️ ComfyUI_LayerStyle
⬇️ ComfyUI-Impact-Pack
⬇️ ComfyUI-KJNodes
⬇️ ComfyUI-SeedVR2_VideoUpscaler
✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)


In [3]:
# CELL 3 — Models: klein-9B + qwen_3_8b + VAE + LoRAs the workflow needs
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
C = '/content/ComfyUI'
for d in ['diffusion_models','text_encoders','vae','loras']:
    os.makedirs(f'{C}/models/{d}', exist_ok=True)

def pull(repo, fname, dest, out_name=None, token=None):
    p = hf_hub_download(repo, fname, local_dir=dest, token=token)
    flat = os.path.join(dest, out_name or os.path.basename(fname))
    if p != flat: shutil.move(p, flat)
    # remove any leftover split_files/ subdir
    sub = os.path.join(dest, fname.split('/')[0])
    if '/' in fname and os.path.isdir(sub): shutil.rmtree(sub, ignore_errors=True)
    return flat

# klein-9B diffusion model (GATED — needs HF_TOKEN + accepted license)  -> workflow node 13
pull('black-forest-labs/FLUX.2-klein-9B', 'flux-2-klein-9b.safetensors', f'{C}/models/diffusion_models', token=HF_TOKEN)
# qwen_3_8b text encoder  -> workflow node 14
pull('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{C}/models/text_encoders', token=HF_TOKEN)
# VAE — workflow node 10 (VAELoader) expects the filename 'flux2-vae-9b.safetensors'
pull('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{C}/models/vae', out_name='flux2-vae-9b.safetensors')

# === LoRAs the uploaded workflow (ai_skin_api.json) references BY EXACT NAME ===
#   node 38 -> flux-2-klein-NSFW.safetensors      node 82 -> Lenovo-UltraReal.safetensors
def grab_lora_file(repo, fname, out_name, revision=None, token=None):
    src = hf_hub_download(repo, fname, local_dir=f'{C}/models/loras', revision=revision, token=token)
    dst = f'{C}/models/loras/{out_name}'
    if src != dst: shutil.move(src, dst)
    print('✅ LoRA:', out_name)

# NSFW LoRA — native Flux.2-klein-9B, pinned to the exact commit you provided
grab_lora_file('msrcam/Flux.2_Klein_9B_LoRas', 'flux2klein_nsfw.safetensors',
               'flux-2-klein-NSFW.safetensors',
               revision='9c8c61dfa7d1314b6d04f946595dcb5865359e02')
# Lenovo UltraReal (Flux2) — HF mirror of the Civitai model (no token needed)
grab_lora_file('Danrisi/Lenovo_UltraReal_Flux2', 'lenovo_flux2.safetensors', 'Lenovo-UltraReal.safetensors')

# klein-enhanced-details is actually used by this workflow (3rd LoRA in the chain, strength 0.3, before KSampler).
# klein-delight is the only genuinely unused leftover LoRA here, kept because it's harmless.
def grab_lora(repo, out_name, prefer=None):
    files = [f for f in list_repo_files(repo)
             if f.endswith('.safetensors') and 'checkpoint' not in f]
    pick = next((f for f in files if prefer and prefer in f), None) or files[-1]
    src = hf_hub_download(repo, pick, local_dir=f'{C}/models/loras')
    dst = f'{C}/models/loras/{out_name}'
    if src != dst: shutil.move(src, dst)
    print(f'✅ LoRA: {out_name}  (from {pick})')
grab_lora('dx8152/Flux2-Klein-9B-Enhanced-Details', 'klein-enhanced-details.safetensors')
grab_lora('linoyts/Flux2-Klein-Delight-LoRA',       'klein-delight.safetensors', prefer='v2')

!ls -lh {C}/models/diffusion_models {C}/models/text_encoders {C}/models/vae {C}/models/loras

flux-2-klein-9b.safetensors: reconstructing file:   0%|          |  0.00B / 18.2GB            

flux-2-klein-9b.safetensors: downloading bytes:           |  0.00B            

split_files/text_encoders/qwen_3_8b_fp8m(…): reconstructing file:   0%|          |  0.00B / 8.66GB            

split_files/text_encoders/qwen_3_8b_fp8m(…): downloading bytes:           |  0.00B            

split_files/vae/flux2-vae.safetensors: reconstructing file:   0%|          |  0.00B /  336MB            

...

realistic.safetensors: downloading bytes:           |  0.00B            

✅ LoRA: klein-enhanced-details.safetensors  (from realistic.safetensors)


pytorch_lora_weights_v2.safetensors: reconstructing file:   0%|          |  0.00B /  127MB            

pytorch_lora_weights_v2.safetensors: downloading bytes:           |  0.00B            

✅ LoRA: klein-delight.safetensors  (from pytorch_lora_weights_v2.safetensors)
/content/ComfyUI/models/diffusion_models:
total 17G
-rw-r--r-- 1 root root 17G Jul 16 19:15 flux-2-klein-9b.safetensors
-rw-r--r-- 1 root root   0 Jul 16 19:11 put_diffusion_model_files_here

/content/ComfyUI/models/loras:
total 781M
-rw-r--r-- 1 root root 159M Jul 16 19:16 flux-2-klein-NSFW.safetensors
-rw-r--r-- 1 root root 121M Jul 16 19:16 klein-delight.safetensors
-rw-r--r-- 1 root root 317M Jul 16 19:16 klein-enhanced-details.safetensors
-rw-r--r-- 1 root root 187M Jul 16 19:16 Lenovo-UltraReal.safetensors
-rw-r--r-- 1 root root    0 Jul 16 19:11 put_loras_here

/content/ComfyUI/models/text_encoders:
total 8.1G
-rw-r--r-- 1 root root    0 Jul 16 19:11 put_text_encoder_files_here
-rw-r--r-- 1 root root 8.1G Jul 16 19:16 qwen_3_8b_fp8mixed.safetensors

/content/ComfyUI/models/vae:
total 321M
-rw-r--r-- 1 root root 321M Jul 16 19:16 flux2-vae-9b.safetensors
-rw-r--r-- 1 root root    0 Jul 16 19:11 put_vae_

In [4]:
# Photo Enhancer workflow — saves to ComfyUI so you can open it from the menu
import json, os
os.makedirs('/content/ComfyUI/user/default/workflows', exist_ok=True)
workflow = {"last_node_id":16,"last_link_id":17,"nodes":[{"id":99,"type":"Note","pos":[50,30],"size":[500,185],"flags":{},"order":0,"mode":0,"inputs":[],"outputs":[],"properties":{"Node name for S&R":"Note"},"widgets_values":["📸 Photo Enhancer — Prompt Bank\n\nCopy the relevant prompt into node 6 (CR Prompt Text):\n\n1 E-COMMERCE: Enhance this product photo: correct and even out the lighting, boost color accuracy and vibrancy, sharpen textures and fine product details, make it look professional and purchase-ready. Do not alter any product features.\n\n2 SOCIAL MEDIA: Enhance this photo for social media: boost colors and contrast, sharpen details, correct exposure and white balance, add a subtle cinematic warmth. Make it visually striking but natural.\n\n3 FOOD: Enhance this food photo: make colors vibrant and appetizing, correct white balance so food looks warm and natural, sharpen fine textures like sauce, steam, and crumbs, make the food look fresh and delicious.\n\n4 REAL ESTATE: Enhance this interior photo: brighten the room naturally, correct mixed lighting and white balance, improve contrast and clarity, make it look inviting and well-lit without overexposure.\n\n5 LANDSCAPE: Enhance this landscape photo: boost sky colors and cloud definition, increase natural color saturation, sharpen distant details, reduce haze. Keep it photorealistic."],"color":"#2a3a4a","bgcolor":"#1a2a3a"},{"id":6,"type":"CR Prompt Text","pos":[430,30],"size":[560,120],"flags":{},"order":5,"mode":0,"inputs":[],"outputs":[{"name":"prompt","type":"STRING","links":[8],"slot_index":0},{"name":"show_help","type":"STRING","links":[],"slot_index":1}],"properties":{"Node name for S&R":"CR Prompt Text"},"widgets_values":["Enhance this product photo: correct and even out the lighting, boost color accuracy and vibrancy, sharpen textures and fine product details, make it look professional and purchase-ready. Do not alter any product features."]},{"id":1,"type":"LoadImage","pos":[50,265],"size":[315,314],"flags":{},"order":1,"mode":0,"inputs":[],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[1],"slot_index":0},{"name":"MASK","type":"MASK","links":[],"slot_index":1}],"properties":{"Node name for S&R":"LoadImage"},"widgets_values":["example.png","image"]},{"id":2,"type":"VAELoader","pos":[50,625],"size":[315,58],"flags":{},"order":2,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[2,16],"slot_index":0}],"properties":{"Node name for S&R":"VAELoader"},"widgets_values":["flux2-vae-9b.safetensors"]},{"id":3,"type":"VAEEncode","pos":[430,375],"size":[210,46],"flags":{},"order":8,"mode":0,"inputs":[{"name":"pixels","type":"IMAGE","link":1,"slot_index":0},{"name":"vae","type":"VAE","link":2,"slot_index":1}],"outputs":[{"name":"LATENT","type":"LATENT","links":[3,4],"slot_index":0}],"properties":{"Node name for S&R":"VAEEncode"}},{"id":4,"type":"CLIPLoader","pos":[50,725],"size":[315,82],"flags":{},"order":3,"mode":0,"inputs":[],"outputs":[{"name":"CLIP","type":"CLIP","links":[5],"slot_index":0}],"properties":{"Node name for S&R":"CLIPLoader"},"widgets_values":["qwen_3_8b_fp8mixed.safetensors","flux2"]},{"id":5,"type":"UNETLoader","pos":[50,855],"size":[315,82],"flags":{},"order":4,"mode":0,"inputs":[],"outputs":[{"name":"MODEL","type":"MODEL","links":[6,7],"slot_index":0}],"properties":{"Node name for S&R":"UNETLoader"},"widgets_values":["flux-2-klein-9b.safetensors","default"]},{"id":7,"type":"CLIPTextEncode","pos":[1050,30],"size":[450,185],"flags":{},"order":9,"mode":0,"inputs":[{"name":"text","type":"STRING","link":8,"widget":{"name":"text"},"slot_index":0},{"name":"clip","type":"CLIP","link":5,"slot_index":1}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[9],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":[]},{"id":8,"type":"ReferenceLatent","pos":[1050,280],"size":[210,46],"flags":{},"order":10,"mode":0,"inputs":[{"name":"conditioning","type":"CONDITIONING","link":9,"slot_index":0},{"name":"latent","type":"LATENT","link":3,"slot_index":1}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[10],"slot_index":0}],"properties":{"Node name for S&R":"ReferenceLatent"}},{"id":9,"type":"BasicGuider","pos":[1310,280],"size":[210,46],"flags":{},"order":11,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":6,"slot_index":0},{"name":"conditioning","type":"CONDITIONING","link":10,"slot_index":1}],"outputs":[{"name":"GUIDER","type":"GUIDER","links":[11],"slot_index":0}],"properties":{"Node name for S&R":"BasicGuider"}},{"id":10,"type":"RandomNoise","pos":[430,625],"size":[315,82],"flags":{},"order":6,"mode":0,"inputs":[],"outputs":[{"name":"NOISE","type":"NOISE","links":[12],"slot_index":0}],"properties":{"Node name for S&R":"RandomNoise"},"widgets_values":[42,"randomize"]},{"id":11,"type":"KSamplerSelect","pos":[430,745],"size":[315,58],"flags":{},"order":7,"mode":0,"inputs":[],"outputs":[{"name":"SAMPLER","type":"SAMPLER","links":[13],"slot_index":0}],"properties":{"Node name for S&R":"KSamplerSelect"},"widgets_values":["euler"]},{"id":12,"type":"BasicScheduler","pos":[430,855],"size":[315,106],"flags":{},"order":8,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":7,"slot_index":0}],"outputs":[{"name":"SIGMAS","type":"SIGMAS","links":[14],"slot_index":0}],"properties":{"Node name for S&R":"BasicScheduler"},"widgets_values":["simple",4,1.0]},{"id":13,"type":"SamplerCustomAdvanced","pos":[1310,450],"size":[210,126],"flags":{},"order":12,"mode":0,"inputs":[{"name":"noise","type":"NOISE","link":12,"slot_index":0},{"name":"guider","type":"GUIDER","link":11,"slot_index":1},{"name":"sampler","type":"SAMPLER","link":13,"slot_index":2},{"name":"sigmas","type":"SIGMAS","link":14,"slot_index":3},{"name":"latent_image","type":"LATENT","link":4,"slot_index":4}],"outputs":[{"name":"output","type":"LATENT","links":[15],"slot_index":0},{"name":"denoised_output","type":"LATENT","links":[],"slot_index":1}],"properties":{"Node name for S&R":"SamplerCustomAdvanced"}},{"id":14,"type":"VAEDecode","pos":[1570,450],"size":[210,46],"flags":{},"order":13,"mode":0,"inputs":[{"name":"samples","type":"LATENT","link":15,"slot_index":0},{"name":"vae","type":"VAE","link":16,"slot_index":1}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[17],"slot_index":0}],"properties":{"Node name for S&R":"VAEDecode"}},{"id":15,"type":"SaveImage","pos":[1830,450],"size":[315,270],"flags":{},"order":14,"mode":0,"inputs":[{"name":"images","type":"IMAGE","link":17,"slot_index":0}],"outputs":[],"properties":{"Node name for S&R":"SaveImage"},"widgets_values":["photo_enhancer/output"]}],"links":[[1,1,0,3,0,"IMAGE"],[2,2,0,3,1,"VAE"],[3,3,0,8,1,"LATENT"],[4,3,0,13,4,"LATENT"],[5,4,0,7,1,"CLIP"],[6,5,0,9,0,"MODEL"],[7,5,0,12,0,"MODEL"],[8,6,0,7,0,"STRING"],[9,7,0,8,0,"CONDITIONING"],[10,8,0,9,1,"CONDITIONING"],[11,9,0,13,1,"GUIDER"],[12,10,0,13,0,"NOISE"],[13,11,0,13,2,"SAMPLER"],[14,12,0,13,3,"SIGMAS"],[15,13,0,14,0,"LATENT"],[16,2,0,14,1,"VAE"],[17,14,0,15,0,"IMAGE"]],"groups":[],"config":{},"extra":{"ds":{"scale":0.7,"offset":[-30,-10]}},"version":0.4}
with open('/content/ComfyUI/user/default/workflows/photo_enhancer_v1.json', 'w') as f:
    json.dump(workflow, f, indent=2)
print('✅ photo_enhancer_v1.json saved — open it in ComfyUI via the top menu')

✅ photo_enhancer_v1.json saved — open it in ComfyUI via the top menu


In [5]:
# =====================================================================
# Install "Save As Script" via the ORIGINAL, actively-maintained extension:
# pydn/ComfyUI-to-Python-Extension (switched from the atmaranto fork after
# its /api/saveascript backend route 500'd on this graph -- traceback would
# be in this same launch cell's output, but the original repo is the safer
# bet: current releases, built for the combined widget/input UI in use here).
# Run this, THEN restart ComfyUI (re-run your launch cell) so it loads.
# =====================================================================
import subprocess, os

CN   = '/content/ComfyUI/custom_nodes'
repo = f'{CN}/ComfyUI-to-Python-Extension'

if not os.path.isdir(repo):
    print('cloning ComfyUI-to-Python-Extension...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/pydn/ComfyUI-to-Python-Extension', repo],
                   check=False)
else:
    print('already present')

# pydn declares deps via pyproject.toml -- editable install picks those up.
# Falls back to requirements.txt too, in case pip can't resolve pyproject here.
subprocess.run(['pip', 'install', '-q', '-e', repo], check=False)
req = f'{repo}/requirements.txt'
if os.path.exists(req):
    subprocess.run(['pip', 'install', '-q', '-r', req], check=False)

print('\n✅ installed. NOW: stop & re-run your ComfyUI launch cell so the node loads.')
print('   Then in the web UI, load your workflow (e.g. ai_skin_api_v3_realneg.json)')
print('   and use File -> Save As Script.')
print('   This version downloads with a fixed filename (workflow_api.py) instead')
print('   of a prompt() dialog -- if it still silently fails, that narrows the bug')
print('   to something graph-specific rather than extension-specific.')

cloning ComfyUI-to-Python-Extension...

✅ installed. NOW: stop & re-run your ComfyUI launch cell so the node loads.
   Then in the web UI, load your workflow (e.g. ai_skin_api_v3_realneg.json)
   and use File -> Save As Script.
   This version downloads with a fixed filename (workflow_api.py) instead
   of a prompt() dialog -- if it still silently fails, that narrows the bug
   to something graph-specific rather than extension-specific.


In [ ]:
# CELL 4 — Launch ComfyUI (embedded inline). KEEP RUNNING. Scroll down in this cell's output for the UI.
import threading, time, socket
%cd /content/ComfyUI
def serve(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0: s.close(); break
        s.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1224)
threading.Thread(target=serve, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server --enable-cors-header

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-07-16 19:16:47.916
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]    0.0 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use
[INFO]    6.2 seconds: /content/ComfyUI/custom_nodes/comfyui

<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data: 5/162
FETCH ComfyRegistry Data: 10/162
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/clipspace.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
FETCH ComfyRegistry Data: 15/162
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui/components/buttonGroup.js. This is likely caused